# RAG for Critical Thinking Churn Analysis
**Google Colab + Free Gemini**

This notebook adds a Retrieval-Augmented Generation (RAG) layer to the churn project.

I use RAG to retrieve relevant internal context (product notes, support themes, past analyses) before generating hypotheses or summaries. This keeps the LLM grounded in project-specific information instead of generic answers.

**Free stack:** Google Colab + Gemini API (AI Studio)

## 1. Setup

In [ ]:
!pip install -q google-generativeai pandas numpy

In [ ]:
import google.generativeai as genai
from getpass import getpass

api_key = getpass("Enter your free Gemini API key (aistudio.google.com): ")
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-1.5-flash")
print("Gemini ready.")

## 2. Synthetic Internal Knowledge Base

In [ ]:
documents = [
    {"id": "doc1", "title": "Product Changelog – UI Redesign", "content": "Major UI redesign released 6 weeks ago. Several users reported difficulty finding core features. Support tickets mentioning navigation issues increased."},
    {"id": "doc2", "title": "Support Theme Summary", "content": "Top themes: confusion after UI update, requests for usage guidance. Tickets from low-usage accounts are disproportionately high."},
    {"id": "doc3", "title": "Previous Churn Analysis", "content": "Customers with monthly usage below 10 hours in the first 60 days had significantly higher 90-day churn. Early engagement was the strongest predictor."},
    {"id": "doc4", "title": "Pricing Change Note", "content": "Price increase applied to ~60% of customers, mainly Basic and Pro plans. Enterprise largely unaffected."},
    {"id": "doc5", "title": "Onboarding Review", "content": "New users often drop off before completing the first key action. Suggested improvements: in-app prompts and a simplified first-week checklist."}
]
print(f"Loaded {len(documents)} documents.")

## 3. Simple Retrieval

In [ ]:
def retrieve(query, docs, top_k=3):
    q = query.lower()
    scored = []
    for doc in docs:
        text = (doc["title"] + " " + doc["content"]).lower()
        score = sum(1 for w in q.split() if w in text)
        scored.append((score, doc))
    scored.sort(reverse=True)
    return [d for s, d in scored[:top_k] if s > 0]

print(retrieve("UI change low usage", documents))

## 4. RAG Query

In [ ]:
def rag(question, docs):
    retrieved = retrieve(question, docs)
    context = "\n\n".join([f"{d['title']}: {d['content']}" for d in retrieved])
    prompt = f"""Use only the following internal documents to answer.

Documents:
{context}

Question: {question}

Answer clearly with bullet points if helpful."""
    return model.generate_content(prompt).text, retrieved

answer, used = rag("What evidence links the UI change to higher churn?", documents)
print(answer)
print("\nUsed:", [d['title'] for d in used])

## 5. Generate Hypotheses & Summary

In [ ]:
print(rag("Generate 4 alternative hypotheses for the churn spike besides price, ranked by document support.", documents)[0])
print("\n---\n")
print(rag("Write a 5-sentence non-technical executive summary of the likely drivers and next actions.", documents)[0])